## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到所在的代码树根 (`solutions/` 或 `tutorials/`)，
   这样 `from attention.mha import ...` 这种导入能直接生效。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd to the tree root (whichever of `solutions/` or `tutorials/` this notebook lives in), turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys

# Walk up from the notebook's CWD until we find a directory named
# `solutions` or `tutorials`. Works no matter which tree the student
# opened. 不论 notebook 位于 solutions/ 还是 tutorials/ 都能正确定位。
ROOTS = {'solutions', 'tutorials'}
if os.path.basename(os.getcwd()) not in ROOTS:
    while os.path.basename(os.getcwd()) not in ROOTS and os.getcwd() != '/':
        os.chdir('..')
    if os.path.basename(os.getcwd()) not in ROOTS:
        # Fallback: maybe we were started at the repo root.
        if os.path.isdir('tutorials'):
            os.chdir('tutorials')
        elif os.path.isdir('solutions'):
            os.chdir('solutions')

assert os.path.basename(os.getcwd()) in ROOTS, (
    f'could not locate solutions/ or tutorials/ from {os.getcwd()}')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the diffusion chapter's reference .pt files live
control_folder = 'diffusion/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 4 章 · Diffusion

## 这是 AF3 与 AF2 最大的不同

AlphaFold 2 的结构头是一个**确定性的**等变 Transformer (Invariant Point Attention + Structure Module)，一次性回归出 3D 坐标。AlphaFold 3 把它**全部丢掉**，换成一个[EDM](https://arxiv.org/abs/2206.00364) 风格的**扩散模型**:

1. **训练**: 给真实坐标加上 σ-量级的 Gaussian 噪声 → 让网络 $F_\theta$ 学会去噪。
2. **推理**: 从纯 Gaussian 噪声 $x \sim \mathcal{N}(0, \sigma_{\max}^2 I)$ 出发，   按 noise schedule 逐步降低 σ、每步调一次 $F_\theta$、用 Euler 步把 $x$ 推向去噪后的位置。

这给 AF3 三个能力 AF2 做不到的:

- **采样多构象**: 从不同初始噪声出发就是不同样本，自然支持"一个序列的多个 3D 解"。
- **配体 / 核酸通用**: 扩散模型在坐标空间通用，不需要为每种分子设计专用 frame。
- **训练更稳**: 去噪目标比一次性回归坐标更易学。

## EDM 数学骨架

EDM 的核心公式 (来自 [Karras et al. 2022](https://arxiv.org/abs/2206.00364)) 把网络 $F_\theta$ 包成 **pre-conditioning**:

$$D_\theta(x; \sigma) = c_\text{skip}(\sigma)\, x + c_\text{out}(\sigma) \, F_\theta\big(c_\text{in}(\sigma)\, x;\, c_\text{noise}(\sigma)\big)$$

其中 $c_\text{skip}$, $c_\text{out}$, $c_\text{in}$ 都是 σ 的简单函数 ——保证不论 σ 多大网络都看到大致单位方差的输入、输出也保持合理量级。

### $c_\text{skip}, c_\text{out}, c_\text{in}$ 的来源 (Karras 2022 eq. 7)

考虑数据分布方差为 $\sigma_\text{data}^2$ (AF3 用 16.0² ≈ 256 Å²)。加噪后的 $x = y + n$，$y \sim p_\text{data}, n \sim \mathcal{N}(0, \sigma^2 I)$。则:

$$\mathrm{Var}(x) = \sigma_\text{data}^2 + \sigma^2$$

为了让网络 $F_\theta$ 看到**单位方差输入**:

$$c_\text{in}(\sigma) = \frac{1}{\sqrt{\sigma_\text{data}^2 + \sigma^2}}$$

Karras 推导了让训练损失 (在 $\sigma$ 上的期望) 满足"信号方差与噪声方差平衡"的 $c_\text{out}, c_\text{skip}$:

$$c_\text{out}(\sigma) = \frac{\sigma \, \sigma_\text{data}}{\sqrt{\sigma_\text{data}^2 + \sigma^2}}, \qquad c_\text{skip}(\sigma) = \frac{\sigma_\text{data}^2}{\sigma_\text{data}^2 + \sigma^2}$$

**直觉**: σ → 0 (噪声小)，$c_\text{skip} → 1, c_\text{out} → 0$，模型几乎直接复用噪声坐标。σ → ∞ (纯噪声)，$c_\text{skip} → 0, c_\text{out} → \sigma_\text{data}$，模型主导输出，残差通道无所谓。

AF3 的具体实现 (`DiffusionModule.forward`):

```python
ratio = sigma / sigma_data            # σ/σ_data
c_skip = 1 / (1 + ratio**2)            # = σ_data² / (σ_data² + σ²)
c_out  = sigma / sqrt(1 + ratio**2)    # 把 sqrt 提出来
c_in   = 1 / sqrt(sigma_data**2 + sigma**2)
x_denoised = c_skip * x_noisy + c_out * F_theta(c_in * x_noisy, sigma)
```

(代码里 $c_\text{noise}$ 通过 `FourierEmbedding(log(σ/σ_data) / 4)` 注入 —— 见第 3 章 §3.2。)

### Noise schedule (推理时的 σ 序列)

推理时从一个**逆向**的 σ 序列采样: $\sigma_T \gg \cdots \gg \sigma_0 = 0$。AF3 用 sigmoid-based schedule，比 DDIM 的几何序列更密集地花预算在中间区域。看 `solutions/diffusion/sampler.py` 里的 `InferenceNoiseScheduler`。

AF3 tiny 模型默认 `N_step=5` (推理便宜)，base 模型 200 步。每多一步都让网络多调一次，
用计算换精度。

## 本章模块

| 文件 | 类 / 函数 | 算法 | 简述 |
|---|---|---|---|
| `diffusion_transformer.py` | `ConditionedTransitionBlock` | 25 | AdaLN-Zero SwiGLU FFN |
| `diffusion_transformer.py` | `DiffusionTransformerBlock` | 23 (单块) | AttentionPairBias + ConditionedTransitionBlock |
| `diffusion_transformer.py` | `DiffusionTransformer` | 23 (整 stack) | 堆叠 n_blocks 个 block |
| `diffusion_module.py` | `DiffusionConditioning` | 21 | 把 trunk + 噪声水平 → (s, z) |
| `diffusion_module.py` | `DiffusionModule.f_forward / forward` | 20 | EDM 包装 + atom→token→atom 主路径 |
| `sampler.py` | `sample_diffusion` | 18 | 完整的 Euler 步采样循环 |
| `frames.py` | `expressCoordinatesInFrame` | 29 | 坐标投影到局部正交基 |
| `model/utils.py` | `centre_random_augmentation` | 19 | recentre + 随机刚体增广 |

本章测试覆盖 4 个 `nn.Module` (CTB / DT block / DT 整 stack / 几何 helper)。顶层 DiffusionModule + sample_diffusion 在 `overview.ipynb` 里端到端跑。

## 4.1 ConditionedTransitionBlock (Algorithm 25)

DiffusionTransformer 每个 block 的 **FFN 分支**。看上去就是个 SwiGLU + AdaLN，但有一个微妙的工程要点 —— 输出门**用的不是 attention 那种 zero-init `linear_o`**，而是再叠一个 sigmoid 门 (`linear_s` 是 BiasInitLinear with biasinit=-2)。

### 完整流程

1. `a = AdaLN(a, s)`  — AdaLN 用 single 条件调制 a
2. `b = SiLU(linear_a1(a)) * linear_a2(a)`  — SwiGLU 在隐藏维 `n*c_a` 上的门控
3. `a = sigmoid(linear_s(s)) * linear_b(b)`  — **adaLN-Zero 输出门** + 投回 c_a

**关键**: 步骤 3 的 sigmoid 门让这个 transition block 起手对残差是 ≈0 贡献(sigmoid(-2)≈0.12)，配合 DropPath 是深层 stack 不爆炸的根本原因。

**任务**: 打开 `diffusion/diffusion_transformer.py`，把 `ConditionedTransitionBlock.forward` 的 TODO 填好。

In [ ]:
from diffusion.diffusion_transformer import ConditionedTransitionBlock
from diffusion.control_values.diffusion_checks import (
    c_a, c_s, c_z, n_heads, n_blocks, test_inputs,
    test_module_shape, test_module_method,
)

ctb = ConditionedTransitionBlock(c_a=c_a, c_s=c_s, n=2, biasinit=-2.0)
test_module_shape(ctb, 'conditioned_transition_block', control_folder)
test_module_method(
    ctb, 'conditioned_transition_block',
    inputs=(test_inputs['a'], test_inputs['s']),
    output_names='out',
    control_folder=control_folder,
    method=lambda a, s: ctb(a=a, s=s),
)
print('ConditionedTransitionBlock ✓')

## 4.2 DiffusionTransformerBlock (Algorithm 23 — 一块)

把 attention 与 FFN 两个分支拼起来:

```text
    a_in    s     z
      \    |    /
       AttentionPairBias(a, s, z)        ← 第 1 章已实现
       │
       DropPath (stochastic depth)
       │
       a_in + ─────► a_mid
                       \
                        ConditionedTransitionBlock(a_mid, s)
                        │
                        DropPath
                        │
                        a_mid + ─────► a_out
```

**两个细节**:

- 返回 `(a_out, s, z)` 而不是只返回 `a_out`。这样 `DiffusionTransformer.forward`  能在 block 之间透传 s/z，而不必每次重新构造 (有助于激活检查点)。
- DropPath 在推理 (eval) 模式下是 nn.Identity()，所以本测试与训练时的输出  会有微小不同 (训练时随机)，但 control values 是在 eval 状态生成的。

**任务**: 在同一个文件里把 `DiffusionTransformerBlock.forward` 的 TODO 填好。

In [ ]:
from diffusion.diffusion_transformer import DiffusionTransformerBlock

def _disable_efficient_attn(mod):
    for m in mod.modules():
        if hasattr(m, 'use_efficient_implementation'):
            m.use_efficient_implementation = False

dtb = DiffusionTransformerBlock(c_a=c_a, c_s=c_s, c_z=c_z, n_heads=n_heads)
_disable_efficient_attn(dtb)
test_module_shape(dtb, 'diffusion_transformer_block', control_folder)
test_module_method(
    dtb, 'diffusion_transformer_block',
    inputs=(test_inputs['a'], test_inputs['s'], test_inputs['z']),
    output_names='a_out',
    control_folder=control_folder,
    method=lambda a, s, z: dtb(a=a, s=s, z=z)[0],
)
print('DiffusionTransformerBlock ✓')

## 4.3 DiffusionTransformer (Algorithm 23 — 整 stack)

把 `n_blocks` 个 DiffusionTransformerBlock 顺序叠起来。看似 1 行 for 循环，但**注意**: 不是单纯 `a = block(a, s, z)`，而是 `a, s, z = block(a, s, z)` ——

三个张量都从 block 出来再喂下一个 block，这样如果未来要加 activation checkpointing(把 s/z 也一并 checkpoint)，只需把这个 for 循环换成 `checkpoint_blocks(...)` 即可。

AtomTransformer (第 1 章 attention/ 末尾) 内部用的也是这个类 ——只是配置 `cross_attention_mode=True` 让 attention 走 cross 路径而非 self。

**任务**: 在同一个文件里填 `DiffusionTransformer.forward`。

In [ ]:
from diffusion.diffusion_transformer import DiffusionTransformer

dt = DiffusionTransformer(
    c_a=c_a, c_s=c_s, c_z=c_z,
    n_blocks=n_blocks, n_heads=n_heads,
)
_disable_efficient_attn(dt)
test_module_shape(dt, 'diffusion_transformer', control_folder)
test_module_method(
    dt, 'diffusion_transformer',
    inputs=(test_inputs['a'], test_inputs['s'], test_inputs['z']),
    output_names='out',
    control_folder=control_folder,
    method=lambda a, s, z: dt(a=a, s=s, z=z),
)
print('DiffusionTransformer ✓')

## 4.4 几何 helper · 局部坐标系 + 刚体增广

AF2 用一整章 (Structure Module) 处理刚体 / 帧 / quaternion，AF3 把这些**塞进扩散**:需要刚体不变性时就在原子坐标空间用几何运算实现。本节实现两个最常用的:

### `expressCoordinatesInFrame` (Algorithm 29)

**Confidence head 计算 PAE (predicted aligned error) 的核心**。给定一组「frame」(每个 frame 由 3 个原子定义) 和一组目标原子，要把每个原子投影到每个 frame 的**局部正交基**上，得到形状 `[..., N_frame, N_atom, 3]` 的相对坐标。

构造正交基的技巧不是 Gram-Schmidt 而是更稳定的版本: 设 a, b, c 是 frame 的三个原子，

$$\mathbf{w}_1 = \widehat{a - b}, \quad \mathbf{w}_2 = \widehat{c - b}$$
$$\mathbf{e}_1 = \widehat{\mathbf{w}_1 + \mathbf{w}_2}, \quad \mathbf{e}_2 = \widehat{\mathbf{w}_2 - \mathbf{w}_1}, \quad \mathbf{e}_3 = \mathbf{e}_1 \times \mathbf{e}_2$$

### 为什么 sum/diff 比 Gram-Schmidt 数值更稳

经典 Gram-Schmidt 写法是: $\mathbf{e}_1 = \hat{\mathbf{w}}_1$, $\mathbf{e}_2 = \widehat{\mathbf{w}_2 - (\mathbf{w}_2 \cdot \mathbf{e}_1) \mathbf{e}_1}$。当 $\mathbf{w}_1$ 和 $\mathbf{w}_2$ **几乎共线** (例如蛋白主链上相邻三个原子接近一条直线)，投影 $\mathbf{w}_2 - (\mathbf{w}_2 \cdot \mathbf{e}_1)\mathbf{e}_1 \to 0$，再 normalize 就放大噪声 ——
**catastrophic cancellation** 经典案例，输出可能跳到完全不相关的方向。

AF3 用的版本 $\mathbf{e}_1 \propto \mathbf{w}_1 + \mathbf{w}_2$, $\mathbf{e}_2 \propto \mathbf{w}_2 - \mathbf{w}_1$:

- 几何上等价于一个 45° 旋转，但用加 / 减替代了 dot + project
- 当 $\mathbf{w}_1 \approx \mathbf{w}_2$，$\mathbf{e}_1$ 仍然有定义  (≈ 2$\mathbf{w}_1$ → normalize 给出 $\hat{\mathbf{w}}_1$)；$\mathbf{e}_2 \to 0$ ——
  确实退化但是**对称**退化，eps 保护下不会爆炸。
- 对 $\mathbf{w}_1, \mathbf{w}_2$ 的小扰动是 Lipschitz 连续的，梯度稳定。

投影就是相对位移 $d = x - b$ 与三个基向量取内积。

### `centre_random_augmentation` (Algorithm 19)

**扩散采样每一步**都先做的事:

1. 减去 (masked) 质心 —— 抵消坐标的整体平移
2. 给每个 sample 各抽一个**随机 SE(3) 变换** (3D 旋转 + 平移)，应用
3. mask 后处理

为什么? AF3 的网络对**全局刚体变换不是天然等变**的 (输入是绝对坐标)。如果不每步增广，模型学到的就是某个固定参考系下的去噪，泛化差。随机增广强迫每次去噪都用一个新的参考系 → 等价于训练目标对刚体变换不敏感。

### 测试只覆盖确定性分支

`centre_random_augmentation` 完整路径要抽 SO(3) 随机数，无法跨平台位级复现；我们测试 `centre_only=True` 分支 (只做减质心)，确定性、可测。

**任务**: 填两个 TODO ——`diffusion/frames.py::expressCoordinatesInFrame` (完整) 和`model/utils.py::centre_random_augmentation` (包括 centre_only + 完整路径)。

In [ ]:
from diffusion.frames import expressCoordinatesInFrame
from model.utils import centre_random_augmentation
from diffusion.control_values.diffusion_checks import test_inputs

# expressCoordinatesInFrame (Algorithm 29)
out = expressCoordinatesInFrame(
    test_inputs['coords'].double(),
    test_inputs['frame_atoms'].double(),
)
expected = torch.load(f'{control_folder}/express_coordinates_in_frame_out.pt')
assert torch.allclose(out, expected), 'expressCoordinatesInFrame output mismatch'
print('expressCoordinatesInFrame ✓')

# centre_random_augmentation, deterministic centre_only=True branch
out = centre_random_augmentation(
    test_inputs['coords'].double(), N_sample=2, centre_only=True,
)
expected = torch.load(f'{control_folder}/centre_random_augmentation_centre_only_out.pt')
assert torch.allclose(out, expected), 'centre_random_augmentation(centre_only=True) output mismatch'
print('centre_random_augmentation (centre_only) ✓')

## 章节小结

本章你实现了 AF3 扩散主干的 5 个核心组件:

| 类 / 函数 | 角色 |
|---|---|
| `ConditionedTransitionBlock` | DiffusionTransformer 的 FFN 分支 (Alg 25) |
| `DiffusionTransformerBlock` | attention + FFN 一块 (Alg 23) |
| `DiffusionTransformer` | n_blocks 块堆叠 |
| `expressCoordinatesInFrame` | PAE 内核 (Alg 29) |
| `centre_random_augmentation` | 采样每步必备的刚体增广 (Alg 19) |

顶层的 `DiffusionConditioning` (Alg 21)、`DiffusionModule.f_forward` / `forward`(Alg 20 EDM scaling)、和 `sample_diffusion` (Alg 18 完整采样循环) **TODO 已写好**但没单测 (它们要一整份特征 dict)。完成本章后这些 TODO 全部能填，在端到端 `overview.ipynb` 跑一次推理就会一次跑通。

**下一站**: 第 5 章 Confidence 把扩散输出的坐标转成 pLDDT / PAE / PDE 等置信度。